# LangChain 高级 Agent 开发学习Demo

本Notebook将深入学习LangChain中的高级Agent开发技术。

## 学习大纲
1. **Agent架构模式** - ReAct、Plan-and-Execute、Self-Ask
2. **Multi-Agent系统** - 多Agent协作、角色分工、通信机制
3. **Agent工具开发** - 自定义工具、工具组合、工具选择策略

## 环境准备

In [ ]:
# 安装依赖
# pip install langchain==0.3.15
# pip install langchain-core==0.3.28
# pip install langchain-community==0.3.14
# pip install dashscope==1.20.11
# pip install wikipedia==1.4.0
# pip install requests==2.32.3
# pip install duckduckgo-search==7.5.5

## 导入库并配置API Key

In [ ]:
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 验证API Key
if DASHSCOPE_API_KEY:
    print("✅ API Key已加载")
else:
    print("❌ 请在.env文件中配置DASHSCOPE_API_KEY")

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 初始化LLM
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.7,
    dashscope_api_key=DASHSCOPE_API_KEY
)

print("✅ LLM初始化完成")

---

# 第一部分：Agent架构模式

## 1.1 ReAct Agent (Reasoning + Acting)

ReAct是最常用的Agent模式，它结合了推理（Reasoning）和行动（Acting）：
- **Thought**: Agent思考下一步该做什么
- **Action**: Agent选择并执行一个工具
- **Observation**: 观察工具执行的结果
- **循环**: 重复上述过程直到得到最终答案

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
import requests
import json

# 定义工具
@tool
def calculate(expression: str) -> str:
    """计算数学表达式的值
    
    Args:
        expression: 数学表达式，如 "2 + 3 * 4"
    
    Returns:
        计算结果
    """
    try:
        # 安全地计算表达式
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"

@tool
def get_current_time(timezone: str = "Asia/Shanghai") -> str:
    """获取当前时间
    
    Args:
        timezone: 时区，默认为Asia/Shanghai
    
    Returns:
        当前时间字符串
    """
    from datetime import datetime
    import pytz
    
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.now(tz)
        return f"当前时间 ({timezone}): {current_time.strftime('%Y-%m-%d %H:%M:%S')}"
    except:
        # 如果时区不支持，使用系统时间
        return f"当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def search_wikipedia(query: str) -> str:
    """在维基百科中搜索信息
    
    Args:
        query: 搜索关键词
    
    Returns:
        搜索结果摘要
    """
    try:
        import wikipedia
        wikipedia.set_lang("zh")
        summary = wikipedia.summary(query, sentences=3)
        return f"维基百科搜索结果:\n{summary}"
    except Exception as e:
        return f"搜索失败: {str(e)}"

print("✅ 工具定义完成")

In [ ]:
# 创建ReAct Agent
tools = [calculate, get_current_time, search_wikipedia]

# 创建提示词模板
react_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个helpful的AI助手。你可以使用以下工具来帮助回答问题:

{tools}

使用ReAct模式思考:
1. Thought: 思考需要做什么
2. Action: 选择并使用工具
3. Observation: 观察结果
4. 重复直到得到答案

请始终使用中文回答。"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 创建Agent
react_agent = create_tool_calling_agent(llm, tools, react_prompt)

# 创建AgentExecutor
react_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

print("✅ ReAct Agent创建完成")

In [ ]:
# 测试ReAct Agent
result = react_executor.invoke({
    "input": "计算 (15 + 25) * 3 的值，然后告诉我现在几点了"
})

print(f"\n{'='*50}")
print(f"最终答案: {result['output']}")
print(f"{'='*50}")

## 1.2 Plan-and-Execute Agent

Plan-and-Execute模式将任务分解为两个阶段：
1. **Planning**: 制定完成任务的步骤计划
2. **Execution**: 按照计划逐步执行

这种模式适合复杂的多步骤任务。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 创建规划器
planner_prompt = ChatPromptTemplate.from_template(
    """你是一个任务规划专家。请将以下任务分解为详细的执行步骤。

任务: {task}

可用工具:
{tools}

请输出执行计划，每个步骤一行，格式如下:
步骤1: [描述]
步骤2: [描述]
...
"""
)

planner_chain = planner_prompt | llm | StrOutputParser()

# 创建执行器提示词
executor_prompt = ChatPromptTemplate.from_template(
    """你是一个任务执行者。请根据以下计划执行当前步骤。

完整计划:
{plan}

当前步骤: {current_step}

之前的执行结果:
{previous_results}

请执行当前步骤并给出结果。
"""
)

print("✅ Plan-and-Execute链创建完成")

In [ ]:
# 测试Plan-and-Execute模式
def plan_and_execute(task: str, tools_desc: str) -> dict:
    """执行Plan-and-Execute流程"""
    
    # 第一步：制定计划
    print("📋 正在制定计划...\n")
    plan = planner_chain.invoke({
        "task": task,
        "tools": tools_desc
    })
    print(f"计划:\n{plan}\n")
    print("="*50)
    
    # 第二步：执行计划
    steps = [s for s in plan.split("\n") if s.strip().startswith("步骤")]
    results = []
    
    for i, step in enumerate(steps, 1):
        print(f"\n🔨 执行: {step}")
        
        # 这里简化处理，实际应该调用Agent执行
        result = executor_prompt.format(
            plan=plan,
            current_step=step,
            previous_results="\n".join(results)
        )
        
        step_result = llm.invoke(result).content
        results.append(f"{step}: {step_result}")
        print(f"结果: {step_result}")
    
    return {
        "plan": plan,
        "results": results
    }

# 测试
tools_description = """1. calculate: 计算数学表达式
2. get_current_time: 获取当前时间
3. search_wikipedia: 搜索维基百科"""

result = plan_and_execute(
    task="研究LangChain框架的历史，并计算它从2022年10月到现在发展了多少个月",
    tools_desc=tools_description
)

print("\n" + "="*50)
print("✅ 任务完成")

---

# 第二部分：Multi-Agent系统

多Agent系统允许创建具有不同专长的Agent，它们可以协作完成复杂任务。

## 2.1 创建专业化Agent

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 定义研究员Agent
class ResearchAgent:
    """研究员Agent - 负责信息收集和研究"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template(
            """你是一个专业的研究员。你的任务是深入研究以下主题并提供详细报告。

研究主题: {topic}

请提供:
1. 主题概述
2. 关键要点
3. 相关发现

研究报告:"""
        )
        self.chain = self.prompt | self.llm | StrOutputParser()
    
    def research(self, topic: str) -> str:
        print(f"🔬 研究员正在研究: {topic}")
        return self.chain.invoke({"topic": topic})

# 定义分析师Agent
class AnalystAgent:
    """分析师Agent - 负责数据分析和洞察"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template(
            """你是一个专业的数据分析师。请分析以下研究报告并提供深入洞察。

研究报告:
{report}

请提供:
1. 关键发现
2. 趋势分析
3. 建议和结论

分析报告:"""
        )
        self.chain = self.prompt | self.llm | StrOutputParser()
    
    def analyze(self, report: str) -> str:
        print(f"📊 分析师正在分析报告...")
        return self.chain.invoke({"report": report})

# 定义撰写者Agent
class WriterAgent:
    """撰写者Agent - 负责内容创作"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template(
            """你是一个专业的技术作家。请基于以下分析报告撰写一篇简洁、易懂的文章。

分析报告:
{analysis}

要求:
1. 结构清晰
2. 语言简洁
3. 突出要点

文章:"""
        )
        self.chain = self.prompt | self.llm | StrOutputParser()
    
    def write(self, analysis: str) -> str:
        print(f"✍️  撰写者正在创作文章...")
        return self.chain.invoke({"analysis": analysis})

print("✅ Multi-Agent系统创建完成")

## 2.2 Agent协作流程

In [ ]:
# 创建Agent实例
researcher = ResearchAgent(llm)
analyst = AnalystAgent(llm)
writer = WriterAgent(llm)

# 定义协作流程
def multi_agent_workflow(topic: str) -> dict:
    """多Agent协作完成任务"""
    
    print(f"\n{'='*60}")
    print(f"🎯 多Agent协作任务开始: {topic}")
    print(f"{'='*60}\n")
    
    # 第一步：研究员收集信息
    research_report = researcher.research(topic)
    print(f"\n📋 研究报告:\n{research_report[:200]}...\n")
    
    # 第二步：分析师分析数据
    analysis_report = analyst.analyze(research_report)
    print(f"\n📈 分析报告:\n{analysis_report[:200]}...\n")
    
    # 第三步：撰写者创作内容
    final_article = writer.write(analysis_report)
    print(f"\n📝 最终文章:\n{final_article}\n")
    
    print(f"{'='*60}")
    print("✅ 多Agent协作任务完成")
    print(f"{'='*60}\n")
    
    return {
        "research": research_report,
        "analysis": analysis_report,
        "article": final_article
    }

# 测试多Agent协作
result = multi_agent_workflow("LangChain在企业AI应用中的优势")

## 2.3 带反馈的Multi-Agent系统

更高级的Multi-Agent系统允许Agent之间相互反馈和迭代。

In [ ]:
class ReviewerAgent:
    """评审员Agent - 负责内容质量审核"""
    
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_template(
            """你是一个严格的内容评审员。请评审以下文章并提供改进建议。

文章:
{article}

请评估:
1. 内容准确性 (1-5分)
2. 结构清晰度 (1-5分)
3. 语言质量 (1-5分)

如果总分 >= 12分，输出"APPROVED"，否则输出"NEEDS_REVISION"并给出具体改进建议。

评审结果:"""
        )
        self.chain = self.prompt | self.llm | StrOutputParser()
    
    def review(self, article: str) -> dict:
        print(f"👁️  评审员正在审核文章...")
        result = self.chain.invoke({"article": article})
        
        approved = "APPROVED" in result.upper()
        return {
            "approved": approved,
            "feedback": result
        }

# 创建带反馈循环的协作流程
def multi_agent_with_feedback(topic: str, max_iterations: int = 2) -> dict:
    """带反馈循环的多Agent协作"""
    
    researcher_agent = ResearchAgent(llm)
    analyst_agent = AnalystAgent(llm)
    writer_agent = WriterAgent(llm)
    reviewer_agent = ReviewerAgent(llm)
    
    print(f"\n{'='*60}")
    print(f"🔄 带反馈的多Agent协作开始: {topic}")
    print(f"{'='*60}\n")
    
    # 研究和分析
    research = researcher_agent.research(topic)
    analysis = analyst_agent.analyze(research)
    
    # 迭代写作和评审
    for iteration in range(max_iterations):
        print(f"\n🔁 迭代 {iteration + 1}/{max_iterations}")
        
        # 撰写
        article = writer_agent.write(analysis)
        
        # 评审
        review_result = reviewer_agent.review(article)
        
        print(f"\n评审结果: {'✅ 通过' if review_result['approved'] else '❌ 需要修订'}")
        print(f"反馈:\n{review_result['feedback'][:200]}...\n")
        
        if review_result['approved']:
            print("\n🎉 文章已通过评审！")
            return {
                "article": article,
                "iterations": iteration + 1,
                "review": review_result
            }
        
        # 如果未通过，更新分析以包含反馈
        analysis = f"{analysis}\n\n评审反馈: {review_result['feedback']}"
    
    print("\n⚠️  达到最大迭代次数")
    return {
        "article": article,
        "iterations": max_iterations,
        "review": review_result
    }

# 测试
result = multi_agent_with_feedback("AI Agent的未来发展趋势", max_iterations=2)

---

# 第三部分：Agent工具开发

## 3.1 自定义高级工具

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Optional, List

# 定义工具输入Schema
class SearchInput(BaseModel):
    """搜索工具的输入参数"""
    query: str = Field(description="搜索查询字符串")
    max_results: int = Field(default=3, description="返回的最大结果数", ge=1, le=10)
    language: str = Field(default="zh", description="搜索语言")

class DataAnalysisInput(BaseModel):
    """数据分析工具的输入参数"""
    data: List[float] = Field(description="要分析的数据列表")
    analysis_type: str = Field(description="分析类型: 'mean', 'median', 'std', 'all'")

# 定义工具函数
def advanced_search(query: str, max_results: int = 3, language: str = "zh") -> str:
    """高级搜索工具"""
    # 这里简化实现
    return f"搜索 '{query}' (语言: {language}, 最多 {max_results} 个结果)\n结果: [示例结果1, 示例结果2, 示例结果3]"

def analyze_data(data: List[float], analysis_type: str) -> str:
    """数据分析工具"""
    import statistics
    
    results = {}
    
    if analysis_type in ["mean", "all"]:
        results["平均值"] = statistics.mean(data)
    
    if analysis_type in ["median", "all"]:
        results["中位数"] = statistics.median(data)
    
    if analysis_type in ["std", "all"]:
        results["标准差"] = statistics.stdev(data) if len(data) > 1 else 0
    
    result_str = "数据分析结果:\n"
    for key, value in results.items():
        result_str += f"{key}: {value:.2f}\n"
    
    return result_str

# 创建结构化工具
advanced_search_tool = StructuredTool.from_function(
    func=advanced_search,
    name="advanced_search",
    description="高级搜索工具，支持多种参数配置",
    args_schema=SearchInput
)

data_analysis_tool = StructuredTool.from_function(
    func=analyze_data,
    name="data_analysis",
    description="数据分析工具，计算统计指标",
    args_schema=DataAnalysisInput
)

print("✅ 高级工具创建完成")

## 3.2 工具组合和工具包

In [ ]:
from typing import List
from langchain_core.tools import BaseTool

class DataScienceToolkit:
    """数据科学工具包"""
    
    @staticmethod
    @tool
    def load_data(file_path: str) -> str:
        """加载数据文件
        
        Args:
            file_path: 文件路径
        """
        return f"已加载数据文件: {file_path}"
    
    @staticmethod
    @tool
    def clean_data(data_description: str) -> str:
        """清洗数据
        
        Args:
            data_description: 数据描述
        """
        return f"已清洗数据: {data_description}"
    
    @staticmethod
    @tool
    def visualize_data(chart_type: str, data_description: str) -> str:
        """可视化数据
        
        Args:
            chart_type: 图表类型 (bar, line, scatter, etc.)
            data_description: 数据描述
        """
        return f"已创建{chart_type}图表，展示: {data_description}"
    
    @classmethod
    def get_tools(cls) -> List[BaseTool]:
        """获取工具包中的所有工具"""
        return [
            cls.load_data,
            cls.clean_data,
            cls.visualize_data,
            data_analysis_tool
        ]

# 创建数据科学Agent
ds_tools = DataScienceToolkit.get_tools()

ds_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个数据科学助手。你可以使用以下工具来帮助进行数据分析:

{tools}

请根据用户需求，合理组合使用这些工具完成数据分析任务。"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

ds_agent = create_tool_calling_agent(llm, ds_tools, ds_prompt)
ds_executor = AgentExecutor(agent=ds_agent, tools=ds_tools, verbose=True)

print("✅ 数据科学Agent创建完成")

In [ ]:
# 测试数据科学Agent
result = ds_executor.invoke({
    "input": "帮我分析一组销售数据 [100, 150, 200, 180, 220, 190]，计算所有统计指标，并创建一个折线图"
})

print(f"\n{'='*50}")
print(f"结果: {result['output']}")
print(f"{'='*50}")

## 3.3 动态工具选择

In [ ]:
class AdaptiveAgent:
    """自适应Agent - 根据任务类型动态选择工具"""
    
    def __init__(self, llm):
        self.llm = llm
        
        # 定义不同领域的工具集
        self.math_tools = [calculate]
        self.time_tools = [get_current_time]
        self.search_tools = [search_wikipedia]
        self.data_tools = DataScienceToolkit.get_tools()
        
        self.tool_categories = {
            "数学计算": self.math_tools,
            "时间查询": self.time_tools,
            "信息搜索": self.search_tools,
            "数据分析": self.data_tools
        }
    
    def select_tools(self, task: str) -> List[BaseTool]:
        """根据任务选择合适的工具"""
        
        # 使用LLM分析任务需要哪些工具
        analysis_prompt = ChatPromptTemplate.from_template(
            """分析以下任务需要哪些类型的工具。

任务: {task}

可用工具类型: {categories}

请只输出需要的工具类型，用逗号分隔，例如: 数学计算,时间查询
"""
        )
        
        analysis = (analysis_prompt | self.llm | StrOutputParser()).invoke({
            "task": task,
            "categories": ", ".join(self.tool_categories.keys())
        })
        
        print(f"📊 工具选择分析: {analysis}\n")
        
        # 收集选中的工具
        selected_tools = []
        for category, tools in self.tool_categories.items():
            if category in analysis:
                selected_tools.extend(tools)
                print(f"✅ 已选择工具类型: {category}")
        
        return selected_tools if selected_tools else list(self.tool_categories.values())[0]
    
    def run(self, task: str):
        """执行任务"""
        print(f"\n🎯 任务: {task}\n")
        
        # 动态选择工具
        tools = self.select_tools(task)
        
        # 创建Agent
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一个智能助手，请使用可用的工具完成任务。"),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),
        ])
        
        agent = create_tool_calling_agent(self.llm, tools, prompt)
        executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=3)
        
        # 执行任务
        result = executor.invoke({"input": task})
        return result

# 创建自适应Agent
adaptive_agent = AdaptiveAgent(llm)

print("✅ 自适应Agent创建完成")

In [ ]:
# 测试自适应Agent
result = adaptive_agent.run(
    "计算从2020年到现在经过了多少年，并分析一下数据 [2020, 2021, 2022, 2023, 2024] 的统计特征"
)

print(f"\n{'='*50}")
print(f"最终结果: {result['output']}")
print(f"{'='*50}")

---

# 总结

## 学到的内容

### 1. Agent架构模式
- ✅ **ReAct模式**: 思考-行动-观察循环，适合大多数场景
- ✅ **Plan-and-Execute模式**: 先规划后执行，适合复杂多步骤任务

### 2. Multi-Agent系统
- ✅ **专业化Agent**: 为不同角色创建专门的Agent
- ✅ **Agent协作**: 通过流程编排实现多Agent协作
- ✅ **反馈循环**: 支持Agent间的反馈和迭代改进

### 3. Agent工具开发
- ✅ **自定义工具**: 使用@tool装饰器和StructuredTool
- ✅ **工具包**: 组织相关工具成工具包
- ✅ **动态工具选择**: 根据任务自适应选择工具

## 最佳实践

1. **工具设计**
   - 工具功能要单一明确
   - 提供清晰的描述和参数说明
   - 使用Pydantic Schema验证输入

2. **Agent设计**
   - 为Agent设定明确的角色和职责
   - 使用合适的提示词引导Agent行为
   - 设置合理的迭代次数限制

3. **Multi-Agent协作**
   - 明确定义Agent间的协作流程
   - 设计有效的通信机制
   - 实现反馈和质量控制机制

## 进阶方向

- 学习LangGraph进行更复杂的Agent编排
- 探索Agent的记忆和状态管理
- 研究Agent的评估和优化方法
- 实现生产级的错误处理和监控